In [1]:
import pandas as pd
df = pd.read_csv("data/ground_truth.csv")
print(df.shape)
print(df.columns.tolist())
print(df.head(3).to_string())

(2931, 13)
['split', 'question_id', 'question', 'sql_query', 'true_answer', 'assumption', 'patient_fhir_id', 'template', 'val_dict', 'proc_query', 'main_table_name', 'mappable_to_fhir', 'true_fhir_ids']
   split               question_id                                                                                                               question                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  sql_query                true_answer                                                              

In [2]:
print(df['split'].value_counts())


split
train    2098
valid     424
test      409
Name: count, dtype: int64


In [3]:
from src.store import get_documents_by_ids
import json

# Agarramos el ID de la primera pregunta
import ast
first_ids = df[df['split']=='test']['true_fhir_ids'].iloc[0]
print(first_ids)

{'Observation': ['8630c91c-cc25-576e-8bcb-6a5af216a4dd', 'be36217c-e272-5783-9d3d-f6c8957bd456', '702218c0-d43b-56cd-b053-f8c67daa6a77', '6ba264de-5f81-5196-ae04-ea60f8ee0dd9', '8f5ae692-f16a-5f4a-bc76-6cc029f2d434', '70f71ed7-945b-54ec-bec4-040748e187d7', 'a115118d-fd35-53bb-8967-93310456d086', 'f4e34ef4-69be-55c5-8cce-f8bac1f6f0ba', '2b1d32e2-b423-54dd-8e86-a8df2bc4eea4', '5db638b5-8e2c-599a-ac4e-7256ddad8870', 'bd1de102-bc3b-5c2c-b36c-31c5c8e8f798', '596528ab-4ad7-58d5-ac3c-0653124905ae', 'b054df07-c451-5aaa-8827-b716d5affe8b', 'f2e03339-f234-5959-9c1c-d9db38faa920']}


In [4]:
from src.store import get_documents_by_ids
import json

doc = get_documents_by_ids(['Observation/8630c91c-cc25-576e-8bcb-6a5af216a4dd'])
print(json.dumps(json.loads(list(doc.values())[0]), indent=2))


{
  "id": "8630c91c-cc25-576e-8bcb-6a5af216a4dd",
  "code": {
    "coding": [
      {
        "code": "50983",
        "system": "http://mimic.mit.edu/fhir/mimic/CodeSystem/mimic-d-labitems",
        "display": "Sodium"
      }
    ]
  },
  "meta": {
    "profile": [
      "http://mimic.mit.edu/fhir/mimic/StructureDefinition/mimic-observation-labevents"
    ]
  },
  "issued": "2136-11-01T11:31:00-04:00",
  "status": "final",
  "subject": {
    "reference": "Patient/28776290-4349-56d3-8c13-adc554feabb8"
  },
  "category": [
    {
      "coding": [
        {
          "code": "laboratory",
          "system": "http://terminology.hl7.org/CodeSystem/observation-category",
          "display": "Laboratory"
        }
      ]
    }
  ],
  "specimen": {
    "reference": "Specimen/176ed7bc-1e8a-55da-b042-c4099abe2ab6"
  },
  "encounter": {
    "reference": "Encounter/ddfdec3b-1cd3-5016-8055-29f764ca34b3"
  },
  "extension": [
    {
      "url": "http://mimic.mit.edu/fhir/mimic/StructureDefiniti

In [5]:
from src.search import search_bm25
result = search_bm25("sodium level patient", n_results=5)
print(result['retrieved_ids'])
print(f"Latency: {result['latency_s']:.2f}s")

/Users/rnorel/.pyenv/versions/3.11.14/envs/CH/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/rnorel/Documents/Learning/Counsel/research-scientist-interview-main/src/search.py:12: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


Building BM25 index (runs once)...
  Loading Condition...


  Loading ConditionED...


  Loading Encounter...


  Loading EncounterED...


...

  Loading ProcedureED...


  Loading ProcedureICU...


  Loading Specimen...


  Loading SpecimenLab...


Loaded 928,935 FHIR resources from data/fhir_records
BM25 index built: 928,935 documents
['Patient/d378a59b-aa80-5bc5-812a-7d59b26e7df4', 'Patient/a5d4cb17-db8d-574b-bd88-71473088fd9a', 'Patient/4f773083-7f4d-5378-b839-c24ca1e15434', 'Patient/86bce885-4646-59f1-8f95-d92545376ce6', 'Patient/91e0e410-0782-5478-90e0-1bedc3aa1525']
Latency: 14.12s


In [6]:
result2 = search_bm25("Sodium", n_results=10)
print(result2['retrieved_ids'])
print(f"Latency: {result2['latency_s']:.2f}s")


['MedicationRequest/b3c44bda-ca9f-5d92-963a-a0588e4c6895', 'MedicationRequest/ee82494e-3588-5d93-8760-f0a32c56aa84', 'MedicationRequest/d40ecf8e-51d9-5f1b-9bcb-20f9a41ca2e1', 'MedicationRequest/e951c6fc-c582-5b1b-b1f9-1378c2523246', 'MedicationRequest/c915431b-ed0d-511a-a71f-6a9123657174', 'MedicationRequest/412bbaa1-2e17-5ea3-8289-0b45eafc502f', 'MedicationRequest/f54a883c-8388-5cb1-a635-b03f4f10d325', 'MedicationRequest/7d83cb4d-9997-517c-8241-d3784e7650e5', 'MedicationRequest/a1f0ac00-d075-5ab5-a1cc-356541dc5912', 'MedicationRequest/1596c27b-a208-50b6-bfb3-1201e59fe7dc']
Latency: 0.12s


In [7]:
from src.data_loader import load_ground_truth

df = load_ground_truth()
test_df = df[df['split'] == 'test'].reset_index(drop=True)

# Ver cómo se ven los true_fhir_ids
print(test_df['true_fhir_ids'].iloc[0])
print(type(test_df['true_fhir_ids'].iloc[0]))
print()
print(test_df['true_fhir_ids'].iloc[1])

{'Observation': ['Observation/8630c91c-cc25-576e-8bcb-6a5af216a4dd', 'Observation/be36217c-e272-5783-9d3d-f6c8957bd456', 'Observation/702218c0-d43b-56cd-b053-f8c67daa6a77', 'Observation/6ba264de-5f81-5196-ae04-ea60f8ee0dd9', 'Observation/8f5ae692-f16a-5f4a-bc76-6cc029f2d434', 'Observation/70f71ed7-945b-54ec-bec4-040748e187d7', 'Observation/a115118d-fd35-53bb-8967-93310456d086', 'Observation/f4e34ef4-69be-55c5-8cce-f8bac1f6f0ba', 'Observation/2b1d32e2-b423-54dd-8e86-a8df2bc4eea4', 'Observation/5db638b5-8e2c-599a-ac4e-7256ddad8870', 'Observation/bd1de102-bc3b-5c2c-b36c-31c5c8e8f798', 'Observation/596528ab-4ad7-58d5-ac3c-0653124905ae', 'Observation/b054df07-c451-5aaa-8827-b716d5affe8b', 'Observation/f2e03339-f234-5959-9c1c-d9db38faa920']}
<class 'dict'>

{}


In [8]:
from src.data_loader import load_ground_truth
from src.search import search_bm25, _build_bm25_index

df = load_ground_truth()
test_df = df[df['split'] == 'test'].reset_index(drop=True)
test_df = test_df[test_df['true_fhir_ids'].apply(lambda x: len(x) > 0)]

# Primera pregunta del test set
row = test_df.iloc[0]
print("QUESTION:", row['question'])
print("TRUE IDS:", row['true_fhir_ids'])

_build_bm25_index()
result = search_bm25(row['question'], n_results=20)
print("RETRIEVED:", result['retrieved_ids'])


QUESTION: Did patient 10025463 receive any lab testing in 11/2136?
TRUE IDS: {'Observation': ['Observation/8630c91c-cc25-576e-8bcb-6a5af216a4dd', 'Observation/be36217c-e272-5783-9d3d-f6c8957bd456', 'Observation/702218c0-d43b-56cd-b053-f8c67daa6a77', 'Observation/6ba264de-5f81-5196-ae04-ea60f8ee0dd9', 'Observation/8f5ae692-f16a-5f4a-bc76-6cc029f2d434', 'Observation/70f71ed7-945b-54ec-bec4-040748e187d7', 'Observation/a115118d-fd35-53bb-8967-93310456d086', 'Observation/f4e34ef4-69be-55c5-8cce-f8bac1f6f0ba', 'Observation/2b1d32e2-b423-54dd-8e86-a8df2bc4eea4', 'Observation/5db638b5-8e2c-599a-ac4e-7256ddad8870', 'Observation/bd1de102-bc3b-5c2c-b36c-31c5c8e8f798', 'Observation/596528ab-4ad7-58d5-ac3c-0653124905ae', 'Observation/b054df07-c451-5aaa-8827-b716d5affe8b', 'Observation/f2e03339-f234-5959-9c1c-d9db38faa920']}
Building BM25 index (runs once)...
  Loading Condition...


  Loading ConditionED...


  Loading Encounter...


  Loading EncounterED...


  Loading EncounterICU...


...

  Loading ProcedureED...


  Loading ProcedureICU...


  Loading Specimen...


  Loading SpecimenLab...


Loaded 928,935 FHIR resources from data/fhir_records
BM25 index built: 928,935 documents
RETRIEVED: ['Observation/5fe18467-f3d8-5ddd-895d-c096b507249c', 'Procedure/9cc1002a-5330-559b-8df6-66e0ded9882b', 'Observation/27e1045a-0c50-5a8c-a3d0-36ae78748f54', 'Observation/bbb622a5-4404-58b9-bb0e-19c419e978e4', 'Observation/b6466a6e-cf68-5e2e-9b9d-d469ed2c5ab7', 'Observation/08ba41e8-669b-566f-80da-d6875a89eca6', 'Observation/cf8958fe-a34c-54b8-851d-a709742ff302', 'Observation/dad482bc-e0ca-502b-9644-97aacaa89057', 'Observation/9b7446ca-e489-5ded-94eb-acb46dc83360', 'Observation/5f9aa093-ea35-53f6-afec-f046235183a8', 'Observation/9a9b15ba-9514-587b-8cdc-16bd70263ab0', 'Observation/ab51daa0-9f53-5919-94e3-3acf636854db', 'Observation/9d44e71b-2fbd-5162-903d-89a0ed156c18', 'Observation/abe0688e-4a17-56cc-aa7d-e1c84d25d101', 'Observation/1fbda863-6058-5cd0-aaf2-15d726f39a15', 'Observation/270b9cbe-aea9-509c-af21-6056d8eee10a', 'Observation/cd6303e6-1636-5a61-bd56-60b214f3bb41', 'Observation/114f

In [9]:
print("QUESTION:", row['question'])
print("TRUE IDS:", row['true_fhir_ids'])

QUESTION: Did patient 10025463 receive any lab testing in 11/2136?
TRUE IDS: {'Observation': ['Observation/8630c91c-cc25-576e-8bcb-6a5af216a4dd', 'Observation/be36217c-e272-5783-9d3d-f6c8957bd456', 'Observation/702218c0-d43b-56cd-b053-f8c67daa6a77', 'Observation/6ba264de-5f81-5196-ae04-ea60f8ee0dd9', 'Observation/8f5ae692-f16a-5f4a-bc76-6cc029f2d434', 'Observation/70f71ed7-945b-54ec-bec4-040748e187d7', 'Observation/a115118d-fd35-53bb-8967-93310456d086', 'Observation/f4e34ef4-69be-55c5-8cce-f8bac1f6f0ba', 'Observation/2b1d32e2-b423-54dd-8e86-a8df2bc4eea4', 'Observation/5db638b5-8e2c-599a-ac4e-7256ddad8870', 'Observation/bd1de102-bc3b-5c2c-b36c-31c5c8e8f798', 'Observation/596528ab-4ad7-58d5-ac3c-0653124905ae', 'Observation/b054df07-c451-5aaa-8827-b716d5affe8b', 'Observation/f2e03339-f234-5959-9c1c-d9db38faa920']}


In [10]:
print(test_df[['question', 'patient_fhir_id']].iloc[0])

question           Did patient 10025463 receive any lab testing i...
patient_fhir_id                 28776290-4349-56d3-8c13-adc554feabb8
Name: 0, dtype: str


In [11]:
import json

with open("data/fhir_records/Patient.ndjson") as f:
    patient = json.loads(f.readline())
    
print(json.dumps(patient, indent=2))

{
  "id": "28dcf33b-0c52-587f-83ad-2a3270976719",
  "meta": {
    "profile": [
      "http://mimic.mit.edu/fhir/mimic/StructureDefinition/mimic-patient"
    ]
  },
  "name": [
    {
      "use": "official",
      "family": "Patient_10007795"
    }
  ],
  "gender": "female",
  "birthDate": "2083-04-10",
  "extension": [
    {
      "url": "http://hl7.org/fhir/us/core/StructureDefinition/us-core-race",
      "extension": [
        {
          "url": "ombCategory",
          "valueCoding": {
            "code": "2106-3",
            "system": "urn:oid:2.16.840.1.113883.6.238",
            "display": "White"
          }
        },
        {
          "url": "text",
          "valueString": "White"
        }
      ]
    },
    {
      "url": "http://hl7.org/fhir/us/core/StructureDefinition/us-core-ethnicity",
      "extension": [
        {
          "url": "ombCategory",
          "valueCoding": {
            "code": "2186-5",
            "system": "urn:oid:2.16.840.1.113883.6.238",
       

In [12]:
import json

# Construir lookup table {numero_clinico: uuid}
patient_id_map = {}

with open("data/fhir_records/Patient.ndjson") as f:
    for line in f:
        patient = json.loads(line)
        uuid = patient.get("id", "")
        for identifier in patient.get("identifier", []):
            clinical_id = identifier.get("value", "")
            if clinical_id:
                patient_id_map[clinical_id] = uuid

print(f"Patients mapped: {len(patient_id_map)}")
print("Example:", list(patient_id_map.items())[:3])

Patients mapped: 100
Example: [('10007795', '28dcf33b-0c52-587f-83ad-2a3270976719'), ('10007928', '74a2fd87-885b-5eca-9f8b-9141915dba51'), ('10009628', '51d2190c-cc46-56c5-b2ea-363895cbea75')]


In [13]:
test_df = df[df['split'] == 'test'].reset_index(drop=True)
print(test_df['mappable_to_fhir'].value_counts())

mappable_to_fhir
True    409
Name: count, dtype: int64


In [14]:
test_df = df[df['split'] == 'test'].reset_index(drop=True)
empty = test_df[test_df['true_fhir_ids'].apply(lambda x: len(x) == 0)]
print(f"Empty true_fhir_ids: {len(empty)}")
print(empty['question'].head(3).tolist())

Empty true_fhir_ids: 109
['Were there any procedures conducted on patient 10006580 during the previous year?', 'Was the mean blood pressure of patient 10014729 ever less than 79.0 on 03/07/this year?', "How many days has it been since patient 10021487's hospital admission?"]


In [15]:
print(empty[['question', 'assumption']].head(5).to_string())


                                                                                                                                                           question                                                                                                                                                        assumption
1                                                                                 Were there any procedures conducted on patient 10006580 during the previous year?                                                                                                                   Assume the current time is 2137-12-31 23:59:00.
2                                                                           Was the mean blood pressure of patient 10014729 ever less than 79.0 on 03/07/this year?                                            Assume the current time is 2125-12-31 23:59:00. Search for 'Arterial Blood Pressure mean' to find mean blood pressure.
7                     

In [16]:
print(empty['true_fhir_ids'].head(5).tolist())

[{}, {}, {}, {}, {}]
